In [ ]:
import os

# change directory 
os.getcwd()
os.chdir("C:\\path\\to\\CNC\\folder\\")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

names = ['CMX1_S_CP2.csv', 'CMX1_AL_CP2.csv', 'CMX1_AL_CP1.csv', 'CMX1_S_CP1.csv', 'DMC2_S_CP2.csv', 'DMC2_AL_CP2.csv', 'DMC2_AL_CP1.csv', 'DMC2_S_CP1.csv']

dfs = {}

for name in names:
    machine, material, component = name.replace('.csv', '').split('_')
    tempdf = pd.read_csv(f"clean_datasets\{name}") 
    tempdf[['Machine', 'Material', 'Component']] = [machine, material, component]
    print(f"Size of {name}: {tempdf.shape}")

    dfs[name.replace('.csv', '')] = tempdf

df = pd.concat(dfs.values(), ignore_index=True)
display(df.head(5))

In [ ]:
# The following code is used to individually observe each dataset (and columns) to set specific thresholds that best fit each sequence. Therefore the variables 'material', 
# 'machine', 'component' need to be specified along with the upper and lower thresholds to visualise it.

In [ ]:
material = 'S'
machine = 'DMC2'
component = 'CP2'

df_filtered = df[df['Material'] == material][df['Machine'] == machine][df['Component'] == component].copy()

# create psuedo labels using threshold method
columns_to_predict = ['CURRENT|1', 'CURRENT|2', 'CURRENT|3', 'CURRENT|6']
window_size = df_filtered.shape[0]  # For plotting

for column in columns_to_predict:
    # Drop missing values and reshape
    data = df_filtered[[column]].dropna()

    upp_perc = data.quantile(0.995)[0]
    low_perc = data.quantile(0.005)[0]

    print(upp_perc, low_perc)
    df_filtered[f'{column}_Peak'] = ((df_filtered[column] > upp_perc) | (df_filtered[column] < low_perc))
    
    # Plot limited number of points
    subset = df_filtered.iloc[:window_size]

    plt.figure(figsize=(12, 6))
    plt.plot(subset[column], label='Current')
    plt.plot(subset.index[subset[f'{column}_Peak']], subset[column][subset[f'{column}_Peak']], 'ro', label='Detected Peak')

    plt.title(f"Threshold based method in {column}")
    plt.xlabel("Index")
    plt.ylabel("Current")
    plt.legend()
    plt.tight_layout
    plt.show()

df_filtered.to_csv(f"datasets_pseudo/Threshold/threshold_{machine}_{material}_{component}.csv")